# ED treatment delay and mortality

## TL;DR

Across all severity levels, the doubly robust AIPW estimate is **+2.15 percentage points** for delay >=160 minutes versus <160 minutes. The simple bootstrap 95% CI is **-0.36 to +6.61 points**, so the signal is not conclusive.

## Context & Methods

- Cohort: all 9,994 visits across severity levels 1-5
- Confounders: severity and age
- Estimator: five-fold cross-fitted propensity-score AIPW
- Uncertainty: 200-replicate percentile bootstrap

### Key assumptions

Consistency, positivity, correct time ordering, and no important unmeasured confounding. AIPW adjusts measured confounding but does not create true randomization; high-delay overlap is sparse in severity levels 4 and 5.

In [1]:
from pathlib import Path
import os
import sys
import pandas as pd

REPO = Path.cwd().resolve()
if not (REPO / "analysis.py").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from analysis import load_data, run_analysis

DATA = Path(os.environ.get("MORTALITY_XLSX", REPO / "data" / "raw" / "mortality.xlsx"))

## Data

In [2]:
data, analysis_data = load_data(DATA)
analysis_data.groupby("Severity_Level").agg(
    visits=("Mortality_Flag", "size"),
    deaths=("Mortality_Flag", "sum"),
).rename_axis("Severity level")

,visits,deaths
Severity level,,
1,461,3
2,1677,25
3,3078,74
4,2902,110
5,1876,120


## Results

The pooled comparison hides the within-severity pattern: observed mortality is higher with high delay in every severity level.

![Crude mortality reversal](../figures/01_crude_mortality_reversal.png)

Severity drives both treatment timing and mortality.

![Severity mechanism](../figures/02_severity_mechanism.png)

In [3]:
result = run_analysis(DATA, REPO, threshold=160, bootstraps=200)
pd.Series({
    "Adjusted risk — high delay (%)": result["risk_high"] * 100,
    "Adjusted risk — lower delay (%)": result["risk_low"] * 100,
    "AIPW risk difference (pp)": result["risk_difference"] * 100,
    "95% CI lower (pp)": result["ci_low"] * 100,
    "95% CI upper (pp)": result["ci_high"] * 100,
}).round(2)

Adjusted risk — high delay (%)     5.37
Adjusted risk — lower delay (%)    3.22
AIPW risk difference (pp)          2.15
95% CI lower (pp)                 -0.36
95% CI upper (pp)                  6.61
dtype: float64

![Doubly robust effect estimate](../figures/03_effect_estimate.png)

## Takeaways

- All severity levels remain in the estimand; severity and age are used for adjustment.
- The adjusted estimate suggests possible harm from long delay.
- The confidence interval includes zero, and overlap is sparse for high delay at levels 4 and 5. This is a signal to investigate, not a clinical threshold.